In [1]:
# imports
import numpy as np
import torch
from pathlib import Path
import sys
from PIL import Image
import cv2 as cv

localImportPath = Path.cwd().parents[2] / 'out/build/x64-release'
if str(localImportPath) not in sys.path:
	sys.path.insert(0,str(localImportPath))	

import touchpy as tp

from diffusers import StableDiffusionXLAdapterPipeline, AutoPipelineForImage2Image, T2IAdapter, EulerAncestralDiscreteScheduler, AutoencoderKL
from diffusers.utils import load_image, make_image_grid

torch.cuda.set_device(0)
frame = 0

c:\Users\Nettoyeur\miniforge3\envs\touchpy\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# iPython interactive shell
from IPython.core.interactiveshell import InteractiveShell

InteractiveShell.ast_node_interactivity = "all"


In [3]:
# pipeline settings and loading models

adapter = T2IAdapter.from_pretrained("TencentARC/t2i-adapter-canny-sdxl-1.0", torch_dtype=torch.float16, variant="fp16").to("cuda")
vae=AutoencoderKL.from_pretrained("madebyollin/sdxl-vae-fp16-fix", torch_dtype=torch.float16)
		
pipe = StableDiffusionXLAdapterPipeline.from_pretrained("stabilityai/sdxl-turbo", vae=vae, adapter=adapter, torch_dtype=torch.float16, variant="fp16")
pipe = pipe.to("cuda")
		
inputBuffer = None
outBuffer = None
testImage = load_image("./idzard.jpg")

Loading pipeline components...: 100%|██████████| 7/7 [00:00<00:00, 13.26it/s]


In [4]:
def on_layout_change(comp, user_data):
    pass

In [11]:
def on_frame(comp, user_data):

	print("frame callback")	
	#read Out TOPs from the tox we loaded
	webcam = comp.out_tops[0].as_tensor()

	
	canny = comp.out_tops[1].as_tensor()
	print(canny.shape)
	######### Process and Copy for next frame (Fast) ###################
	comp.start_next_frame()	
	

	"""
	prompt = "Mystical fairy in real, magic, 4k picture, high quality"
	negative_prompt = "extra digit, fewer digits, cropped, worst quality, low quality, glitch, deformed, mutated, ugly, disfigured"

	result = this.pipe(
		prompt=prompt,
		negative_prompt=negative_prompt,
		image=canny,
		width=512,
		height=512,
		output_type="pt"
	).images[0]	
	result
	#print(result.device)
	#comp.in_tops[0].from_tensor(result)
	"""

	frame += 1

In [10]:
frame

0

In [6]:
# set tox to load
comp = tp.Comp("tox/yolo.tox", flags=tp.CompFlags.INTERNAL_TIME_ASYNC)
comp.set_on_layout_change_callback(on_layout_change, {})
comp.set_on_frame_callback(on_frame, {})

In [7]:
comp.start()

In [ ]:
comp.stop()

In [ ]:
comp.unload()
del(comp)